In [ ]:
!pip install tensorflow nltk scikit-learn

#Importing libraries


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GRU,
    Dense,
    Dropout,
    Bidirectional
)

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

#Loading dataset

In [ ]:
df = pd.read_csv("/content/mbti_1.csv")

df.head()

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...


understanding data


In [ ]:
df.shape

(8675, 2)

In [ ]:
df['type'].value_counts()

,count
type,
INFP,1832
INFJ,1470
INTP,1304
INTJ,1091
ENTP,685
ENFP,675
ISTP,337
ISFP,271
ENTJ,231


In [ ]:
df['IE'] = df['type'].apply(lambda x: 0 if x[0] == 'I' else 1)

df['NS'] = df['type'].apply(lambda x: 0 if x[1] == 'N' else 1)

df['TF'] = df['type'].apply(lambda x: 0 if x[2] == 'T' else 1)

df['JP'] = df['type'].apply(lambda x: 0 if x[3] == 'J' else 1)

In [ ]:
df[['type','IE','NS','TF','JP']].head()

,type,IE,NS,TF,JP
0,INFJ,0,0,1,0
1,ENTP,1,0,0,1
2,INTP,0,0,0,1
3,INTJ,0,0,0,0
4,ENTJ,1,0,0,0


In [ ]:
def clean_text(text):

    text = text.lower()

    # remove urls
    text = re.sub(r'http\S+', '', text)

    # remove non letters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
df['clean_posts'] = df['posts'].apply(clean_text)


In [ ]:
print(df['posts'][0][:500])

print("\n\nAFTER CLEANING:\n")

print(df['clean_posts'][0][:500])

'http://www.youtube.com/watch?v=qsXHcwe3krw|||http://41.media.tumblr.com/tumblr_lfouy03PMA1qa1rooo1_500.jpg|||enfp and intj moments  https://www.youtube.com/watch?v=iz7lE1g4XM4  sportscenter not top ten plays  https://www.youtube.com/watch?v=uCdfze1etec  pranks|||What has been the most life-changing experience in your life?|||http://www.youtube.com/watch?v=vXZeYwwRDw8   http://www.youtube.com/watch?v=u8ejam5DP3E  On repeat for most of today.|||May the PerC Experience immerse you.|||The last thin


AFTER CLEANING:

and intj moments sportscenter not top ten plays pranks what has been the most life changing experience in your life on repeat for most of today may the perc experience immerse you the last thing my infj friend posted on his facebook before committing suicide the next day rest in peace enfj sorry to hear of your distress it s only natural for a relationship to not be perfection all the time in every moment of existence try to figure the hard times as times of growth as welcome

In [ ]:
vocab_size = 10000

In [ ]:
tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(df['clean_posts'])

In [ ]:
sequences = tokenizer.texts_to_sequences(df['clean_posts'])

In [ ]:
max_len = 500

In [ ]:
X = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post',
    truncating='post'
)

In [ ]:
X.shape

(8675, 500)

Here we have run the model for target IE(introvert/extrovert critiriea)

In [ ]:
y_ie = df['IE']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_ie,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

In [ ]:
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print(class_weights)

{0: np.float64(0.6518880330640616), 1: np.float64(2.1459492888064315)}


In [ ]:
IE_model = Sequential()

IE_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_len
    )
)

IE_model.add(
    Bidirectional(
        GRU(64)
    )
)

IE_model.add(Dropout(0.5))

IE_model.add(Dense(32, activation='relu'))

IE_model.add(Dense(1, activation='sigmoid'))

In [ ]:
IE_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
IE_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = IE_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights
)

Epoch 1/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 121s 661ms/step - accuracy: 0.4669 - loss: 0.6941 - val_accuracy: 0.3451 - val_loss: 0.7008
Epoch 2/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 117s 667ms/step - accuracy: 0.6156 - loss: 0.6671 - val_accuracy: 0.5115 - val_loss: 0.7696
Epoch 3/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 112s 643ms/step - accuracy: 0.7882 - loss: 0.4652 - val_accuracy: 0.5576 - val_loss: 0.8168
Epoch 4/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 108s 620ms/step - accuracy: 0.9215 - loss: 0.2146 - val_accuracy: 0.6405 - val_loss: 0.9399
Epoch 5/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 112s 644ms/step - accuracy: 0.9694 - loss: 0.0837 - val_accuracy: 0.6016 - val_loss: 1.5697


In [ ]:
loss, accuracy = IE_model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - accuracy: 0.6282 - loss: 1.4803
Accuracy: 0.6282420754432678


In [ ]:
def predict_ie(text):

    # clean text
    text = clean_text(text)

    # convert to sequence
    seq = tokenizer.texts_to_sequences([text])

    # padding
    padded = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post',
        truncating='post'
    )

    # prediction
    pred = IE_model.predict(padded)[0][0]

    if pred >= 0.5:
        personality = "Extrovert"
    else:
        personality = "Introvert"

    confidence = round(max(pred, 1-pred) * 100, 2)

    return personality, confidence

In [ ]:
text = """
I enjoy spending time alone reading books
and thinking deeply about ideas.
"""

result = predict_ie(text)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 483ms/step
('Introvert', np.float32(99.8))


In [ ]:
text = """
I love attending parties, meeting new people,
and being surrounded by friends. Social interactions
make me feel energetic and excited.
"""

result = predict_ie(text)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
('Introvert', np.float32(97.29))


In [ ]:
IE_model.save("IE_model.h5")

In [ ]:
print(history.history['accuracy'])
print(history.history['val_accuracy'])

[0.46685880422592163, 0.615634024143219, 0.7881844639778137, 0.9214697480201721, 0.9693803787231445]
[0.34510084986686707, 0.5115273594856262, 0.5576369166374207, 0.640489935874939, 0.6015850305557251]


creating a question so necessary data can be pulled out from user

In [ ]:
questions = [

    "How do you usually spend your weekends?",

    "What kind of social situations make you comfortable?",

    "Do you prefer planning things or being spontaneous?",

    "How do you make important decisions?",

    "What type of environment helps you focus best?",

    "How do you react after a long stressful day?",

    "Would you rather work in a team or alone?",

    "Do you enjoy deep conversations or casual chats more?",

    "How do you usually prepare before an event?",

    "What motivates you more: logic or emotions?"
]

In [ ]:
responses = []

for q in questions:

    ans = input(q + "\n")

    responses.append(ans)

How do you usually spend your weekends?
alone
What kind of social situations make you comfortable?
single
Do you prefer planning things or being spontaneous?

How do you make important decisions?

What type of environment helps you focus best?

How do you react after a long stressful day?

Would you rather work in a team or alone?

Do you enjoy deep conversations or casual chats more?

How do you usually prepare before an event?

What motivates you more: logic or emotions?



In [ ]:
full_conversation = " ".join(responses)

In [ ]:
result = predict_ie(full_conversation)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
('Introvert', np.float32(99.98))


here we will train another model based on NS(Intution/sensing)

In [ ]:
y_ns = df['NS']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_ns,
    test_size=0.2,
    random_state=42
)

In [ ]:
NS_model = Sequential()
NS_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    )
)

NS_model.add(
    Bidirectional(
        GRU(64)
    )
)

NS_model.add(Dropout(0.3))

NS_model.add(Dense(32, activation='relu'))

NS_model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
NS_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
NS_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = NS_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 130s 706ms/step - accuracy: 0.8601 - loss: 0.4199 - val_accuracy: 0.8610 - val_loss: 0.4017
Epoch 2/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 140s 699ms/step - accuracy: 0.8651 - loss: 0.3415 - val_accuracy: 0.8523 - val_loss: 0.4507
Epoch 3/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 124s 711ms/step - accuracy: 0.9193 - loss: 0.1964 - val_accuracy: 0.8285 - val_loss: 0.6793
Epoch 4/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 141s 704ms/step - accuracy: 0.9764 - loss: 0.0676 - val_accuracy: 0.8069 - val_loss: 0.8882
Epoch 5/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 121s 696ms/step - accuracy: 0.9912 - loss: 0.0249 - val_accuracy: 0.7586 - val_loss: 1.2034


In [ ]:
loss, accuracy = NS_model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 135ms/step - accuracy: 0.7769 - loss: 1.1596
Accuracy: 0.7769452333450317


In [ ]:
def predict_ns(text):

    # clean text
    text = clean_text(text)

    # convert to sequence
    seq = tokenizer.texts_to_sequences([text])

    # padding
    padded = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post',
        truncating='post'
    )

    # prediction
    pred = NS_model.predict(padded)[0][0]

    if pred >= 0.5:
        personality = "Sensing"
    else:
        personality = "Intution"

    confidence = round(max(pred, 1-pred) * 100, 2)

    return personality, confidence

In [ ]:
text = """
I often think about future possibilities,
abstract ideas, and innovative concepts.
I enjoy imagining different outcomes and exploring theories.
"""

result = predict_ns(text)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 505ms/step
('Intution', np.float32(99.85))


In [ ]:
text = """
I focus on practical details and real-world experiences.
I prefer facts over theories and like following
proven methods instead of experimenting.
"""

result = predict_ns(text)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
('Intution', np.float32(74.99))


In [ ]:
NS_model.save("NS_model.h5")

here we are training the data based on TF(thinking/feeling)

In [ ]:
y_tf = df['TF']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_tf,
    test_size=0.2,
    random_state=42
)

In [ ]:
TF_model = Sequential()

TF_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    )
)

TF_model.add(
    Bidirectional(
        GRU(64)
    )
)

TF_model.add(Dropout(0.3))

TF_model.add(Dense(32, activation='relu'))

TF_model.add(Dense(1, activation='sigmoid'))

In [ ]:
TF_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
TF_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = TF_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 135s 745ms/step - accuracy: 0.5373 - loss: 0.6889 - val_accuracy: 0.5576 - val_loss: 0.6812
Epoch 2/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 132s 758ms/step - accuracy: 0.6511 - loss: 0.6345 - val_accuracy: 0.5793 - val_loss: 0.6986
Epoch 3/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 136s 780ms/step - accuracy: 0.8273 - loss: 0.3906 - val_accuracy: 0.5814 - val_loss: 0.7841
Epoch 4/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 129s 740ms/step - accuracy: 0.9361 - loss: 0.1618 - val_accuracy: 0.5836 - val_loss: 1.2105
Epoch 5/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 138s 719ms/step - accuracy: 0.9800 - loss: 0.0598 - val_accuracy: 0.5865 - val_loss: 1.6367


In [ ]:
loss, accuracy = TF_model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

55/55 ━━━━━━━━━━━━━━━━━━━━ 6s 115ms/step - accuracy: 0.5781 - loss: 1.6125
Accuracy: 0.5780979990959167


In [ ]:
def predict_tf(text):

    # clean text
    text = clean_text(text)

    # convert to sequence
    seq = tokenizer.texts_to_sequences([text])

    # padding
    padded = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post',
        truncating='post'
    )

    # prediction
    pred = TF_model.predict(padded)[0][0]

    if pred >= 0.5:
        personality = "feelings"
    else:
        personality = "thinking"

    confidence = round(max(pred, 1-pred) * 100, 2)

    return personality, confidence

In [ ]:
text = """
I usually make decisions based on logic and objective analysis.
I try to remain rational even in emotionally difficult situations.
"""

result = predict_tf(text)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 493ms/step
('thinking', np.float32(95.64))


In [ ]:
text = """
I care deeply about people’s emotions and relationships.
When making decisions, I consider how others might feel.
"""

result = predict_tf(text)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
('feelings', np.float32(71.25))


In [ ]:
TF_model.save("TF_model.h5")

here we are training JS(judging /perceving)

In [ ]:
y_jp = df['JP']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_jp,
    test_size=0.2,
    random_state=42
)

In [ ]:
JP_model = Sequential()

JP_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    )
)

JP_model.add(
    Bidirectional(
        GRU(64)
    )
)

JP_model.add(Dropout(0.3))

JP_model.add(Dense(32, activation='relu'))

JP_model.add(Dense(1, activation='sigmoid'))

In [ ]:
JP_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
JP_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = JP_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 127s 696ms/step - accuracy: 0.5973 - loss: 0.6754 - val_accuracy: 0.6117 - val_loss: 0.6671
Epoch 2/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 146s 719ms/step - accuracy: 0.6652 - loss: 0.6058 - val_accuracy: 0.5807 - val_loss: 0.7231
Epoch 3/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 123s 705ms/step - accuracy: 0.8433 - loss: 0.3562 - val_accuracy: 0.5454 - val_loss: 0.9162
Epoch 4/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 123s 710ms/step - accuracy: 0.9469 - loss: 0.1408 - val_accuracy: 0.5461 - val_loss: 1.4146
Epoch 5/5
174/174 ━━━━━━━━━━━━━━━━━━━━ 121s 695ms/step - accuracy: 0.9789 - loss: 0.0590 - val_accuracy: 0.5403 - val_loss: 2.1324


In [ ]:
loss, accuracy = JP_model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

55/55 ━━━━━━━━━━━━━━━━━━━━ 8s 142ms/step - accuracy: 0.5326 - loss: 2.0193
Accuracy: 0.5325648188591003


In [ ]:
def predict_jp(text):

    # clean text
    text = clean_text(text)

    # convert to sequence
    seq = tokenizer.texts_to_sequences([text])

    # padding
    padded = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post',
        truncating='post'
    )

    # prediction
    pred = JP_model.predict(padded)[0][0]

    if pred >= 0.5:
        personality = "Perceving"
    else:
        personality = "Judging"

    confidence = round(max(pred, 1-pred) * 100, 2)

    return personality, confidence

In [ ]:
text = """
I like planning everything in advance and following schedules.
I feel comfortable when tasks are organized and structured.
"""

result = predict_jp(text)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 497ms/step
('Perceving', np.float32(98.78))


In [ ]:
text = """
I prefer flexibility and spontaneity in life.
I usually make decisions at the last moment and enjoy adapting to situations naturally.
"""

result = predict_jp(text)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
('Perceving', np.float32(74.7))


In [ ]:
JP_model.save("JP_model.h5")

In [ ]:
import pickle

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [ ]:
text1 = """
I enjoy staying alone and reading books quietly.
Crowds drain my energy.
"""

In [ ]:
text2 = """
I love parties and social gatherings.
Meeting new people excites me.
"""

In [ ]:
seq1 = tokenizer.texts_to_sequences([clean_text(text1)])

pad1 = pad_sequences(seq1, maxlen=max_len)

print(IE_model.predict(pad1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
[[0.01630356]]


In [ ]:
seq2 = tokenizer.texts_to_sequences([clean_text(text2)])

pad2 = pad_sequences(seq2, maxlen=max_len)

print(IE_model.predict(pad2))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
[[0.01084588]]


In [ ]:
print(df['IE'].value_counts())

IE
0    6676
1    1999
Name: count, dtype: int64


In [ ]:
print(df['TF'].value_counts())

TF
1    4694
0    3981
Name: count, dtype: int64


In [ ]:
print(df['NS'].value_counts())

NS
0    7478
1    1197
Name: count, dtype: int64


In [ ]:
print(df['JP'].value_counts())

JP
1    5241
0    3434
Name: count, dtype: int64
